# Example: Todo App

The main element of the app is a **task widget**. We will extend our previous [simple implementation](/topics/apps/01-flet.html#composite-controls) of this into a full, feature-rich application. The general idea is the same, every interactive control calls a hook that affects the app state and consequently the UI elements (e.g. visibility, focus) which we build imperatively.

## Initial version

Start by initializing the app:

```bash
$ uv run flet create
The app has been created.

Run the app:

flet run
```

This gives us an initial template:

```bash
$ tree .
.
├── README.md
├── pyproject.toml
└── src
    ├── assets
    │   ├── icon.png
    │   └── splash_android.png
    └── main.py

3 directories, 5 files
```

In a real project, you will first have to edit details in the `pyproject.toml` file and `README.md`. For our purposes, we will focus on the `main.py` file. We add the following code:

```{.python filename=src/v1.py}
import flet as ft

def main(page: ft.Page):
    def add_clicked(e):
        task_list.controls.append(ft.Checkbox(label=new_task.value))
        new_task.value = ""     # blank = show hint text again
        main_col.update()

    new_task = ft.TextField(
        hint_text="What needs to be done?", 
        expand=True, 
        on_submit=add_clicked   # ENTER triggers on_submit
    )
    add_button = ft.FloatingActionButton(
        icon=ft.Icons.ADD, 
        on_click=add_clicked
    )
    
    new_task_row = ft.Row(controls=[new_task, add_button])
    task_list = ft.Column()
    main_col = ft.Column(width=600, controls=[new_task_row, task_list])
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(main_col)


if __name__ == "__main__":
    ft.run(main)
```

Reading this backwards, we see that the main control is `main_col` which is centered horizontally. The main column contains the text field for adding a task followed by the task list. The task list is a column whose controls are appended with new task items that are implemented as **checkbox**.

<video
  src="./img/flet-todo/todo-v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The main hook here is `add_clicked` on the ADD button. Moreover, pressing ENTER on the keyboard triggers the same hook. This appends a checkbox to the task list column, resets the text field to blank (which makes the text hint show), and finally updates the main control.

## Refactoring into a control subclass

Note that the app packages nicely as a single composite control with methods (e.g. the hooks). Moreover the app manages its data as class attributes. The following behaves exactly as the first one:

```{.python filename=src/v2.py}
import flet as ft

class TodoApp(ft.Column):
    def __init__(self, width: int):
        super().__init__(width=width)
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.new_task_row = ft.Row(controls=[self.new_task, self.add_button])
        self.task_list = ft.Column()
        self.controls.extend([self.new_task_row, self.task_list])

    def add_clicked(self, e):
        self.task_list.controls.append(ft.Checkbox(label=self.new_task.value))
        self.new_task.value = ""    # blank = show hint text again
        self.update()


def main(page: ft.Page):
    todo = TodoApp(width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(todo)


if __name__ == "__main__":
    ft.run(main)
```

This feels like a cleaner more maintainable implementation of the original app.

## Task items as composite controls

Each **task item** is a composite column control consisting of two views. The default visible view is `display_view` which is a row consisting of three controls: (1) a checkbox with the **task name** (`.label`) and a clickable **toggle** (the state can be accessed via the a boolean attribute `.value`), (2) an edit button which hooks the `.edit_clicked(e)` method, and (3) a delete button which hooks the `.delete_clicked(e)` method. Next, we have the `edit_view` which is similarly a row with `TextField` and a save button which hooks `.save_clicked(e)`. 

![](./img/flet-todo/task_item.drawio.png)

^Actually the checkbox doesn't extend as in the figure. Otherwise, the ff. demo shows the buttons behavior:

<video
  src="./img/flet-todo/todo-v3.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The definition of each hook can be seen in the code:

```{.python filename=src/v3.py}
import flet as ft
from typing import Callable

class TaskItem(ft.Column):
    def __init__(self, text: str, delete_hook: Callable):
        super().__init__()
        self.delete_hook = delete_hook
        self.checkbox = ft.Checkbox(label=text)
        
        # buttons
        self.edit_button = ft.IconButton(
            icon=ft.Icons.EDIT,
            on_click=self.edit_clicked
        )

        self.delete_button = ft.IconButton(
            icon=ft.Icons.DELETE,
            on_click=self.delete_clicked
        )

        self.save_button = ft.IconButton(
            icon=ft.Icons.SAVE,
            on_click=self.save_clicked
        )

        # two views
        self.display_view = ft.Row(controls=[
            self.checkbox, self.edit_button, self.delete_button
        ])

        self.edit_view = ft.Row(
            controls=[
                ft.TextField(
                    value=self.checkbox.label, 
                    expand=True, 
                    on_submit=self.save_clicked
                ),
                self.save_button,
            ], 
            visible=False
        )

        self.controls.extend([self.display_view, self.edit_view])

    def delete_clicked(self, e):
        """Remove this task from the todo list using external hook."""
        self.delete_hook(self)

    def edit_clicked(self, e):
        self.display_view.visible = False
        self.edit_view.visible = True
        self.update()

    def save_clicked(self, e):
        self.checkbox.label = self.edit_view.controls[0].value
        self.display_view.visible = True
        self.edit_view.visible = False
        self.update()

    def is_isolated(self):
        return True
...
```

Observe that the hooks explicitly flips the visibility of the two views so that only one view is visible at each time. Clicking edit, makes only the `edit_view` visible. This makes the edit field visible and clicking save does the reverse while replacing the checkbox label to the contents of the `TextField`. Finally, delete task signals the delete hook which calls an [external]{.underline} function which makes sense since its the outer app container which handles the list of `TaskItem`. Let us now proceed with the `TodoApp` which maintains this said list:

```{.python filename=src/v3.py}
...

class TodoApp(ft.Column):
    def __init__(self, page: ft.Page, width: int):
        super().__init__(width=width)
        self._page = page
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.task_list = ft.Column()
        self.controls.extend([
            ft.Row(controls=[self.new_task, self.add_button]), 
            self.task_list
        ])

    def add_clicked(self, e):
        task = TaskItem(self.new_task.value, delete_hook=self.delete_task)
        self.task_list.controls.append(task)
        self.new_task.value = ""    # blank = show hint text again
        self.update()

    def is_isolated(self):
        return True
    
    def delete_task(self, task: TaskItem):
        dlg_modal = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete this task?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.task_list.controls.remove(task), 
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(dlg_modal)
        self._page.update()
        

def main(page: ft.Page):
    todo = TodoApp(page, width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD), todo)

if __name__ == "__main__":
    ft.run(main)
```

You can see that the final app is simply the previous version a column containing a text field for new tasks and a column containing the task items. Hence, in this version, we added the task item functionality. 

Additionally, we have the `delete_task` function which opens a **delete modal** and removes a task from the list. Note that `.remove` is $O(n)$ but since the task list is <100 or <1000 at the very extreme (probably this has to be enforced in an actual app), it should be fine. The `delete_task` function is inserted into each `TaskItem` which calls `delete_task(self)` allowing the main `TodoApp` to know which task to delete from its list.

:::{.callout-note}
Here we used `TextButton` for "No" and `Button` for "Yes" in the delete modal. However, it's not apparent from the code that we only selected one over the other because of **styling** reasons &mdash; the latter is elevated while the former blends into the background. It might be better to make styling explicit and just use `Button`.
:::

## Final enhancements: active tasks, tab filters, focus